# Nb high-index surface-cell generation with pyHECTR

This notebook generates surface oriented bulk unit cells for the Nb surfaces

- Nb(326)
- Nb(438)
- Nb(539)
- Nb(6 4 11)


The crystallographic transformation, coordinate enumeration, atom count
checking, and structure file writing are implemented in the package module.

The generated structure is a complete periodic bulk cell written in a surface coordinate system: the first two cell vectors are in the surface plane and the third vector points toward the chosen outward surface normal. This is the coordinate system needed for high precision `.xtl` inspection files and ROD `.bul` input files.

The final coordinates are generated directly from an exact integer transformation matrix so that large rectangular high index cells are not limited by rounded exported fractional coordinates.

## 1. Import 

In [1]:
from pathlib import Path
import numpy as np
from pyhectr.surface import (
    cell_parameters_from_vectors,
    generate_surface_cell,
    generate_surface_files,
    write_bul,
    write_xtl,
)

## 2. Coordinate convention

`pyhectr.surface` stores Cartesian lattice vectors as rows:

$$
A =
\begin{bmatrix}
\mathbf a \\
\mathbf b \\
\mathbf c
\end{bmatrix}.
$$

For an integer transformation matrix $P$, the **columns** of $P$ contain the coefficients of the new surface-cell vectors in the original bulk basis:

$$
P =
\begin{bmatrix}
| & | & | \\
\mathbf a'_{\mathrm{bulk}} & \mathbf b'_{\mathrm{bulk}} & \mathbf c'_{\mathrm{bulk}} \\
| & | & |
\end{bmatrix}.
$$


With this convention,

$$A_{\mathrm{surface}} = P^T A_{\mathrm{bulk}},$$

and fractional coordinates are transformed internally as


$$
\mathbf f_{\mathrm{surface}} = (\mathbf f_{\mathrm{bulk}} + \mathbf n - \mathbf p)P^{-T},
$$


where $\mathbf n$ is an integer bulk lattice translation and $\mathbf p$ is an optional origin shift in the old bulk fractional basis. This is the same role as the origin vector in a VESTA transformation. If no origin shift is given, $\mathbf p=(0,0,0)$.

The resulting fractional coordinates $(x',y',z')$ are surface coordinates. The $x'$ and $y'$ directions are periodic in the surface plane. The $z'$ direction follows the transformed third lattice vector, which is chosen to point toward the positive surface normal. 

For ROD bulk files, the same periodic cell can be shifted by one repeat along $z'$ so that the semi infinite bulk lies below the surface reference plane.

## 3. Define the bulk Nb structure

Nb is represented by its body-centered cubic cell.

The basis contains two atoms:

$$(0,0,0),\qquad\left(\frac12,\frac12,\frac12\right).$$

A single species string, `species="Nb"`, is sufficient because
`generate_surface_cell` broadcasts it to all basis positions.

In [2]:
# Nb lattice parameter [Å].
a0_nb = 3.3004

# Cartesian lattice vectors stored as rows: [a_vector, b_vector, c_vector].
bulk_lattice = np.eye(3) * a0_nb

# Fractional coordinates of the bcc basis.
bulk_fractional = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.5, 0.5, 0.5],
    ],
    dtype=float,
)

bulk_lattice, bulk_fractional

(array([[3.3004, 0.    , 0.    ],
        [0.    , 3.3004, 0.    ],
        [0.    , 0.    , 3.3004]]),
 array([[0. , 0. , 0. ],
        [0.5, 0.5, 0.5]]))

## 4. Surface transformation matrices


The columns correspond to the transformed $\mathbf a'$, $\mathbf b'$, and
$\mathbf c'$ vectors. 

For Nb cells, the third column is parallel to
the desired surface normal. The determinant gives the volume multiplication factor, because the bcc Nb cell contains two atoms, the transformed cell should
contain:

$$
N_{\mathrm{atoms}} = 2\,|\det(P)|.
$$

In [3]:
SURFACE_TRANSFORMS = {
    # Nb(326)
    "326": np.array(
        [
            [2, 2, 3],
            [-15, 0, 2],
            [4, -1, 6],
        ],
        dtype=int,
    ),

    # Nb(438)
    "438": np.array(
        [
            [-2, 3, 4],
            [0, -20, 3],
            [1, 6, 8],
        ],
        dtype=int,
    ),

    # Nb(539)
    "539": np.array(
        [
            [0, -6, 5],
            [3, 1, 3],
            [-1, 3, 9],
        ],
        dtype=int,
    ),

    # Nb(6 4 11)
    "6411": np.array(
        [
            [33, -2, 6],
            [22, 3, 4],
            [-26, 0, 11],
        ],
        dtype=int,
    ),
}

for label, transform in SURFACE_TRANSFORMS.items():
    volume_factor = int(round(abs(np.linalg.det(transform))))
    expected_atoms = len(bulk_fractional) * volume_factor
    print(
        f"Nb({label:>4}) | det(P) = {volume_factor:4d} "
        f"| expected atoms = {expected_atoms:4d}"
    )


Nb( 326) | det(P) =  245 | expected atoms =  490
Nb( 438) | det(P) =  445 | expected atoms =  890
Nb( 539) | det(P) =  230 | expected atoms =  460
Nb(6411) | det(P) = 2249 | expected atoms = 4498


## 5. Generate and write the transformed cells

For each surface, the package function performs the complete supercell
enumeration automatically.

Three files are written for each orientation:

- `.xtl`: P1 structure for inspection in VESTA;
- `.bul`: ROD bulk file shifted by one repeat along fractional \(z\);
- `_positive.bul`: the same cell without the \(z=-1\) shift.

The ROD shift is controlled by `z_shift=-1.0`. .

`sort_by_z=True` only changes the order in which atoms are written to the
`.bul` file.

In [4]:
output_dir = Path("surface_cells")

results = {}

for label, transform in SURFACE_TRANSFORMS.items():
    results[label] = generate_surface_files(
        label=label,
        transform=transform,
        lattice_vectors=bulk_lattice,
        fractional_coordinates=bulk_fractional,
        species="Nb",
        output_dir=output_dir,
        file_stem=f"Nb_{label}",
        title=f"Niobium ({label})",
        comment=f"# Niobium ({label})",
        save_xtl=True,
        save_bul=True,
        save_positive_bul=True,
    )

    a, b, c, alpha, beta, gamma = results[label]["cell_parameters"]
    n_atoms = len(results[label]["fractional"])

    print(
        f"Nb({label:>4}) | atoms = {n_atoms:4d} | "
        f"a = {a:10.4f} Å | b = {b:10.4f} Å | c = {c:10.4f} Å | "
        f"angles = ({alpha:.2f}, {beta:.2f}, {gamma:.2f})°"
    )

print(f"\nFiles written to: {output_dir}")

Nb( 326) | atoms =  490 | a =    51.6594 Å | b =     7.3799 Å | c =    23.1028 Å | angles = (90.00, 90.00, 90.00)°
Nb( 438) | atoms =  890 | a =     7.3799 Å | b =    69.6220 Å | c =    31.1359 Å | angles = (90.00, 90.00, 90.00)°
Nb( 539) | atoms =  460 | a =    10.4368 Å | b =    22.3844 Å | c =    35.3928 Å | angles = (90.00, 90.00, 90.00)°
Nb(6411) | atoms = 4498 | a =   156.5169 Å | b =    11.8998 Å | c =    43.4100 Å | angles = (90.00, 90.00, 90.00)°

Files written to: surface_cells


## 6. Check the transformed lattice vector orientation

For these particular cubic Nb transformations, the three transformed lattice
vectors were chosen to be mutually orthogonal.

Because the original Nb cell is cubic, checking the dot products of the
columns of $P$ is sufficient to verify this construction.

A value of zero means the corresponding transformed directions are
perpendicular.

In [5]:
for label, transform in SURFACE_TRANSFORMS.items():
    a_coeff = transform[:, 0]
    b_coeff = transform[:, 1]
    c_coeff = transform[:, 2]

    surface_normal = tuple(int(value) for value in c_coeff)

    print(
        f"Nb({label:>4}) | "
        f"a'·b' = {np.dot(a_coeff, b_coeff):4d} | "
        f"a'·c' = {np.dot(a_coeff, c_coeff):4d} | "
        f"b'·c' = {np.dot(b_coeff, c_coeff):4d} | "
        f"surface-normal coefficients = {surface_normal}"
    )


Nb( 326) | a'·b' =    0 | a'·c' =    0 | b'·c' =    0 | surface-normal coefficients = (3, 2, 6)
Nb( 438) | a'·b' =    0 | a'·c' =    0 | b'·c' =    0 | surface-normal coefficients = (4, 3, 8)
Nb( 539) | a'·b' =    0 | a'·c' =    0 | b'·c' =    0 | surface-normal coefficients = (5, 3, 9)
Nb(6411) | a'·b' =    0 | a'·c' =    0 | b'·c' =    0 | surface-normal coefficients = (6, 4, 11)


## 7. Inspect one generated structure

All generated data are retained in the `results` dictionary.

For example, the Nb(539) entry contains its transformed lattice,
fractional coordinates, species labels, and cell parameters.

In [6]:
surface = results["539"]

print("Nb(539) lattice vectors [Å]:")
print(surface["lattice"])

print("\nCell parameters (a, b, c, alpha, beta, gamma):")
print(surface["cell_parameters"])

print("\nNumber of atoms:")
print(len(surface["fractional"]))

print("\nFirst 10 fractional coordinates:")
print(surface["fractional"][:10])


Nb(539) lattice vectors [Å]:
[[  0.       9.9012  -3.3004]
 [-19.8024   3.3004   9.9012]
 [ 16.502    9.9012  29.7036]]

Cell parameters (a, b, c, alpha, beta, gamma):
(10.436781189619719, 22.384401876306633, 35.39284699483781, 90.0, 90.0, 90.0)

Number of atoms:
460

First 10 fractional coordinates:
[[0.         0.86956522 0.04347826]
 [0.3        0.89130435 0.06956522]
 [0.2        0.95652174 0.14782609]
 [0.7        0.84782609 0.0173913 ]
 [0.6        0.91304348 0.09565217]
 [0.5        0.97826087 0.17391304]
 [0.9        0.93478261 0.12173913]
 [0.1        0.67391304 0.00869565]
 [0.         0.73913043 0.08695652]
 [0.4        0.69565217 0.03478261]]


## 8. Using the same API for other materials


For a single element structure, pass one species string:

```python
surface_lattice, surface_fractional, surface_species = generate_surface_cell(
    lattice_vectors,
    fractional_coordinates,
    transform,
    species="Si",
)
```

For a multi element basis, pass one species label per basis position:

```python
surface_lattice, surface_fractional, surface_species = generate_surface_cell(
    lattice_vectors,
    fractional_coordinates,
    transform,
    species=["Cs", "Cl"],
)
```

The lattice may be non-cubic and non-orthogonal; only a nonsingular
`(3, 3)` set of Cartesian lattice vectors is required.